# Enhanced Combined Drought Index - Precipitation Drought Index

* **Products used:** 
[rainfall_chirps_daily](https://explorer.digitalearth.africa/products/rainfall_chirps_daily),
[ls5_sr](https://explorer.digitalearth.africa/products/ls5_sr),
[ls7_sr](https://explorer.digitalearth.africa/products/ls7_sr),
[ls8_sr](https://explorer.digitalearth.africa/products/ls8_sr),
[ls9_sr](https://explorer.digitalearth.africa/products/ls9_st)

## Background 

Drought is an extended period, during which, fresh water availability and accessibility for the ecosystem at a given time and place is below normal, due to unfavourable spatial and temporal distribution of rainfall, temperature, soil moisture and wind characteristics [(Balint et al., 2013)](https://doi.org/10.1016/B978-0-444-59559-1.00023-2). Severe droughts can affect large populations, leading to a long-term threat to people’s livelihoods and result in tremendous economic loss [(Enenkel et al., 2016)](https://doi.org/10.3390/rs8040340).

The Enhanced Combined Drought Index aims at the timely and reliable detection of drought events with regard to their spatio-temporal extent and severity. The Enhanced Drought Index is a combination of the following: a precipitation component, which considers rainfall deficits and dryness persistence; a soil moisture component, which considers soil moisture deficits and deficit persistence; a vegetation component which considers NDVI deficits and deficit persistence; and a temperature component, which considers temperature excesses and persistence of high temperatures. The index uses satellite-derived rainfall, soil moisture, land surface temperature and vegetation status as input datasets. [(Enenkel et al., 2016)](https://doi.org/10.3390/rs8040340)

The drought index calculated using the precipitation component is referred as the **Precipitation Drought Index (PDI)**, while the index based on temperature is named as **Temperature Drought Index (TDI)**, the index based on soil moisture is named as the **Soil Moisture Drought Index** and that based on the vegetation component is named as **Vegetation Drought Index (VDI)**.

Each drought index can be expressed as:

$\text{Drought Index} = \frac{\text{Actual Average for Interest Period}}{\text{Long Term Average for Interest Period}} * \sqrt{\frac{\text{Actual Length of Continuous Deficit or Excess in the Interest Period}}{\text{Long Term Average of Continuous Deficit or Excess in the Interest Period}}}$

Each drought index is calculated similarly. The equation below illustrates the calculation of the ECDI precipitation component:

\begin{equation}
\text{PDI}_{y,d} = \frac{
\frac{1}{\text{IP}} \sum_{j=0}^{\text{IP} - 1} P^*_{y,(d-j)}}{\frac{1}{n}\sum_{k=1}^n[\frac{1}{\text{IP}} \sum_{j=0}^{\text{IP} - 1} P^*_{(d-j), k}]} * \sqrt{\frac{(\text{RL}^*)P^*_{d, y}}{\frac{1}{n}\sum_{k=1}^{n}(\text{RL}^*)P^*_{d, k}}}
\end{equation}

- $\text{PDI}_{y,d}$ is the Precipitation Drought Index for year $\text{y}$ and time unit (dekad/month) $\text{d}$

- $P^*$ is the modified dekadal/monthly precipitation average 

- $\text{RL}*$ is the modified run length parameter 

- $\text{RL*}(P*)$ (run length) is the maximum number of successive dekads/months below the long-term average rainfall in the interest period
> **Note**: For temperature, run length is the maximum number of successive dekads/months above the long-term average temperature in the interest period

- $\text{IP}$ is the interest period (e.g. 3, 4, 5, . . . dekads/months) (longer IPs detect more severe drought events). IP is flexible defines to what extent past observations are considered.

- $n$ is the number of years where relevant data are available,

- $j$ is the summation running parameter covering the Interest Period

- $k$ is the summation parameter covering the years where relevant data are available

- $d$ time unit (dekad or month) 

- $y$ year

The raw time series of temperature and precipitation as well as the run length are modified to adjust the range of all variables and to avoid a division by zero.

\begin{equation}
T^* = (T_{max} + 1) - T
\end{equation}

\begin{equation}
P^* = P + 1
\end{equation}

\begin{equation}
\text{RL}^* = (\text{RL}_{max} + 1) - \text{RL}
\end{equation}

\begin{equation}
\text{NDVI}^* = \text{NDVI} - (\text{NDVI}_{min} -0.01)
\end{equation}


- $T^*$ is the modified dekadal/monthly temperature average 
- $P^*$ is the modified dekadal/monthly precipitation average 
- $\text{RL}^*$ is the modified run length

All the individual drought indices differ in range. To improve their interpretability and visual comparability a simple scaling factor is introduced.

\begin{equation}
PDI_{scaled} = \frac{(PDI - PDI_{min})}{(PDI_{max} - PDI_{min})}
\end{equation}

The weight of each individual drought index is automatically calculated for every grid point (pixel) with respect to its capability to reflect the future vegetation status (NDVI) and multiplied by the respective individual index to calculate the ECDI. In the case of data gaps in one input dataset, the weights are automatically redistributed to other available variables.

\begin{equation}
ECDI = \sum_{i-1}^{n}w_{i} * \text{DI}_{i}
\end{equation}

- $ECDI$ Enhanced Combined Drought Index 

- $w$ Weight for each individual drought index (e.g., rainfall)

- $\text{DI}$ Individual drought index 

- $n$ number of drought indices used to calculate the ECDI 

- $i$ running parameter covering the number of drought indices

\begin{equation}
w_{i} = \frac{\frac{lag^*_{i}}{\sum_{j=1}^{n} lag^*_{j}} + \frac{corr^*_{i}}{\sum_{j=1}^{n} corr^*_{j}}}{2}
\end{equation}

- $w$ weight for the respective drought index 

- $lag^*$ modified time lag for the respective parameter 

- $corr^*$ modified correlation coefficient for the respective parameter 

- $i$ index for the respective parameter/drought index 

- $j$ running parameter covering all parameters used for the ECDI calculation

- $n$ number of individual drought indices used for the ECDI calculation

## Description



The notebook outlines:

***

## Getting started

To run this analysis, run all the cells in the notebook, starting with the "Load packages" cell. 

### Load packages

In [1]:
import calendar
import os
from datetime import datetime, timedelta

import dask
import geopandas as gpd
import numpy as np
import pandas as pd
import toolz
import xarray as xr
from datacube import Datacube
from deafrica_tools.bandindices import calculate_indices
from deafrica_tools.dask import create_local_dask_cluster
from deafrica_tools.datahandling import load_ard
from odc.geo.geom import Geometry

### Set up a Dask cluster

Dask can be used to better manage memory use and conduct the analysis in parallel. 
For an introduction to using Dask with Digital Earth Africa, see the [Dask notebook](../../Beginners_guide/08_Parallel_processing_with_dask.ipynb).

>**Note**: We recommend opening the Dask processing window to view the different computations that are being executed; to do this, see the *Dask dashboard in DE Africa* section of the [Dask notebook](../../Beginners_guide/08_Parallel_processing_with_dask.ipynb).

To use Dask, set up the local computing cluster using the cell below.

In [2]:
create_local_dask_cluster()

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/victoria@kartoza.com/proxy/8787/status,
Dashboard: /user/victoria@kartoza.com/proxy/8787/status,Workers: 1
Total threads: 15,Total memory: 97.21 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:46071,Workers: 1
Dashboard: /user/victoria@kartoza.com/proxy/8787/status,Total threads: 15
Started: Just now,Total memory: 97.21 GiB
Comm: tcp://127.0.0.1:42029,Total threads: 15
Dashboard: /user/victoria@kartoza.com/proxy/44089/status,Memory: 97.21 GiB
Nanny: tcp://127.0.0.1:42543,


### Analysis parameters

The following cell sets important parameters for the analysis:

* `vector_file`: file path for the shapefile containing the area of interest polygon.
* `resolution`: The x and y cell resolution of the satellite data in metres (spatial resolution). We'll use 5,000 m, which is approximately equal to the default CHIRPS resolution.
* `output_crs` : The coordinate reference system that the loaded data is to be reprojected to.
* `dask_chunks`:  the size of the dask chunks, dask breaks data into manageable chunks that can be easily stored in memory, e.g. `dict(x=1000,y=1000)`
* `time_range` : Time range to load data for.
* `ip` : The interest period to use to calculate the drought indices e.g. (3, 4, 5 dekads). It defines to what extent past observations are considered. Longer IPs detect more severe drought events. For example, if the IP=3 dekads, then the drought index (say PDI) of 0.35 for dekad 2 of 2006 implies actual drought for dekad 36 of 2005, dekad 1 of 2006 and dekad 2 of 2006.
* `output_dir` :  The directory in which to store results from the analysis.

**If running the notebook for the first time**, keep the default settings below.
This will demonstrate how the analysis works and provide meaningful results.

In [3]:
vector_file = "https://raw.githubusercontent.com/Mondieki/kenya-counties-subcounties/master/geojson/baringo.json"
resolution = (-5000, 5000)
output_crs = "EPSG:6933"
dask_chunks = dict(x=3200,y=3200)
time_range = ("2014","2024")
output_dir = "results"
# Corresponding to the six-months Standardized Precipitation Evapotranspiration Index
ip = 18

In [4]:
# Create the outpur directory if it does not exist.
os.makedirs(output_dir, exist_ok=True)

In [5]:
# Load the area of interest
aoi_gdf = gpd.read_file(vector_file)
geopolygon = Geometry(geom=aoi_gdf.geometry.iloc[0], crs=aoi_gdf.crs)

### Connect to the datacube

Connect to the datacube so we can access DE Africa data.
The `app` parameter is a unique name for the analysis which is based on the notebook file name.

In [6]:
# Connect to the datacube
dc = Datacube(app="DroughtIndex")

## Load CHIRPS data

Load the CHIRPS rainfall data from the datacube using the analysis parameters set in the previous section.

In [7]:
%%time
# Load the daily rainfall  data
ds_rf_daily = dc.load(product="rainfall_chirps_daily",
                      measurements=['rainfall'],
                      geopolygon=geopolygon,
                      time=time_range,
                      resolution=resolution,
                      output_crs=output_crs,
                      dask_chunks=dask_chunks
                     )
ds_rf_daily

CPU times: user 38.4 s, sys: 9.75 s, total: 48.2 s
Wall time: 38.2 s


<xarray.Dataset>
Dimensions:      (time: 3682, y: 49, x: 20)
Coordinates:
  * time         (time) datetime64[ns] 2014-01-01T11:59:59.500000 ... 2024-03...
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
Data variables:
    rainfall     (time, y, x) float32 dask.array<chunksize=(1, 49, 20), meta=np.ndarray>
Attributes:
    crs:           EPSG:6933
    grid_mapping:  spatial_ref

In [8]:
# Convert to DataArray
da_rf_daily = ds_rf_daily["rainfall"]
# Convert missing values to nan
da_rf_daily = da_rf_daily.where(da_rf_daily != da_rf_daily.nodata)

da_rf_daily

<xarray.DataArray 'rainfall' (time: 3682, y: 49, x: 20)>
dask.array<where, shape=(3682, 49, 20), dtype=float32, chunksize=(1, 49, 20), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 2014-01-01T11:59:59.500000 ... 2024-03...
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
Attributes:
    units:         mm
    nodata:        -9999
    crs:           EPSG:6933
    grid_mapping:  spatial_ref

## Load Landsat surface reflectance data
Load the Landsat surface reflectance data from the datacube using the analysis parameters set in the previous section and the measurements used to calculate the **NDVI** spectral index.

In [9]:
%%time
# Load the Landsat data
ds_ls_sr = load_ard(dc=dc,
                    products=["ls5_sr", "ls7_sr", "ls8_sr", "ls9_sr"],
                    measurements=["nir", "red"],
                    time=time_range,
                    dask_chunks=dask_chunks,
                    like=ds_rf_daily.geobox)
ds_ls_sr

Using pixel quality parameters for USGS Collection 2
Finding datasets
    ls5_sr
    ls7_sr
    ls8_sr
    ls9_sr
Applying pixel quality/cloud mask
Re-scaling Landsat C2 data
Returning 1966 time steps as a dask array
CPU times: user 3.46 s, sys: 158 ms, total: 3.61 s
Wall time: 6.48 s


<xarray.Dataset>
Dimensions:      (time: 1966, y: 49, x: 20)
Coordinates:
  * time         (time) datetime64[ns] 2014-01-01T07:45:00.396578 ... 2024-05...
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
Data variables:
    nir          (time, y, x) float32 dask.array<chunksize=(1, 49, 20), meta=np.ndarray>
    red          (time, y, x) float32 dask.array<chunksize=(1, 49, 20), meta=np.ndarray>
Attributes:
    crs:           PROJCS["WGS 84 / NSIDC EASE-Grid 2.0 Global",GEOGCS["WGS 8...
    grid_mapping:  spatial_ref

### Calculate the NDVI spectral index

In [10]:
# Calculate the NDVI spectral index.
ds_ndvi = calculate_indices(ds_ls_sr, index=['NDVI'], satellite_mission='ls', drop=True)
# Convert to DataArray
da_ndvi = ds_ndvi["NDVI"]

da_ndvi

Dropping bands ['nir', 'red']


<xarray.DataArray 'NDVI' (time: 1966, y: 49, x: 20)>
dask.array<truediv, shape=(1966, 49, 20), dtype=float32, chunksize=(1, 49, 20), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 2014-01-01T07:45:00.396578 ... 2024-05...
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933

## Resample data

Resample the rainfall `da_rf` and NDVI `da_ndvi` timeseries into **dekadal** (10-day) timesteps.

In [11]:
def get_dekad(date: np.datetime64) -> np.datetime64:
    """
    Get the start date of the dekad that a date belongs to.
    Every month has three dekads, such that the first two dekads
    have 10 days (i.e., 1-10, 11-20), and the third is comprised of the
    remaining days of the month.

    Parameters
    ----------
    date : np.datetime64
        Date to check.

    Returns
    -------
    np.datetime64
        Start date of the dekad.
    """
    timestamp = pd.Timestamp(date)
    year = timestamp.year
    month = timestamp.month

    first_day = datetime(year, month, 1)
    last_day = datetime(year, month, calendar.monthrange(year, month)[1])

    d1_start_date, d2_start_date, d3_start_date = pd.date_range(
        start=first_day, end=last_day, freq="10D", inclusive="left"
    )

    if d1_start_date <= timestamp < d2_start_date:
        return np.datetime64(d1_start_date, "ns")
    elif d2_start_date <= timestamp < d3_start_date:
        return np.datetime64(d2_start_date, "ns")
    else:
        return np.datetime64(d3_start_date, "ns")

def group_by_dekad(
    ds: xr.DataArray | xr.Dataset,
) -> xr.core.groupby.DataArrayGroupBy | xr.core.groupby.DatasetGroupBy:
    """
    Group a dataset or array by dekad.

    Parameters
    ----------
    ds : xr.DataArray | xr.Dataset
        Dataset or DataArray to group

    Returns
    -------
    xr.core.groupby.DataArrayGroupBy | xr.core.groupby.DatasetGroupBy
        Groupby oject
    """
    group = xr.DataArray(
        data=np.vectorize(get_dekad)(ds.time.values),
        coords=ds.time.coords,
        dims=ds.time.dims,
        name="dekad",
        attrs=ds.time.attrs,
    )
    grouped_by_dekad = ds.groupby(group)
    return grouped_by_dekad

In [12]:
%%time
# Resample the rainfall data from daily to decadal (10-day) intervals
da_rf_dekadal = group_by_dekad(da_rf_daily).mean(dim="time").compute()
da_rf_dekadal

/usr/local/lib/python3.10/dist-packages/rasterio/warp.py:344: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  _reproject(


CPU times: user 15.4 s, sys: 397 ms, total: 15.8 s
Wall time: 1min 8s


<xarray.DataArray 'rainfall' (dekad: 363, y: 49, x: 20)>
array([[[ 0.29200163,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.29984912,  0.28720802,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.33419874,  0.31537917,  0.28235275, ...,  0.        ,
          0.        ,  0.        ],
        ...,
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.34356213,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ]],

       [[ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
...
        [ 2.5407748 ,  2.4794157 ,  2.8383396 , ...,  1.5651367 ,
          1.4777172 ,  0.8918543 ],
        [ 2.2201219 ,  2.3179328 ,  2.2798665 , ...,  1.5410626 ,
          1.9449116 ,  1.8832439 ],
        [ 2.5376573 ,  2.7804077 ,  2.3720465 , ...,  1.6634972 ,
          2.1893876 ,  2.3906493 ]],

       [[ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          2.2162352 ,  0.        ],
        ...,
        [ 9.424622  , 10.19459   , 10.981411  , ...,  0.        ,
          1.0213484 ,  0.        ],
        [ 9.818012  ,  9.534503  ,  9.844227  , ...,  1.1844184 ,
          1.1683893 ,  1.1354309 ],
        [10.010099  , 10.197859  , 10.551265  , ...,  1.6940008 ,
          1.6322978 ,  1.3778231 ]]], dtype=float32)
Coordinates:
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21
Attributes:
    units:         mm
    nodata:        -9999
    crs:           EPSG:6933
    grid_mapping:  spatial_ref

In [13]:
%%time
# Resample the NDVI data from daily to decadal (10-day) intervals
da_ndvi_dekadal = group_by_dekad(da_ndvi).mean(dim="time")
da_ndvi_dekadal

CPU times: user 1.14 s, sys: 46.5 ms, total: 1.19 s
Wall time: 1.14 s


<xarray.DataArray 'NDVI' (dekad: 373, y: 49, x: 20)>
dask.array<stack, shape=(373, 49, 20), dtype=float32, chunksize=(1, 49, 20), chunktype=numpy.ndarray>
Coordinates:
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-05-11

In [14]:
# Ensure the NDVI timeseries has the same timesteps as the rainfall timeseries.
da_ndvi_dekadal = da_ndvi_dekadal.sel(dekad=da_rf_dekadal.dekad).compute()
assert all(da_rf_dekadal.dekad == da_ndvi_dekadal.dekad)
da_ndvi_dekadal

<xarray.DataArray 'NDVI' (dekad: 363, y: 49, x: 20)>
array([[[0.5246651 , 0.21412478, 0.36771566, ..., 0.07224146,
         0.10696262, 0.21577993],
        [0.49482203, 0.35397673, 0.4083935 , ..., 0.07264111,
         0.07292355, 0.22460452],
        [0.5497342 , 0.35644487, 0.30848923, ..., 0.05420848,
         0.18062873, 0.21955493],
        ...,
        [0.8086616 , 0.1386993 , 0.24276157, ...,        nan,
         0.4080112 , 0.5302966 ],
        [0.63616383, 0.19208694,        nan, ..., 0.5026036 ,
         0.44152564, 0.5900986 ],
        [       nan, 0.5727446 , 0.35779616, ..., 0.55022687,
         0.48454323, 0.66430485]],

       [[0.54182345, 0.25571483, 0.2746559 , ..., 0.05868618,
         0.10060515, 0.13064624],
        [0.40749332, 0.2912916 ,        nan, ..., 0.06116824,
         0.06439868, 0.16005345],
        [0.4827307 , 0.31603828, 0.3043204 , ..., 0.07225259,
         0.10217287, 0.13892531],
...
        [0.52958417, 0.6212574 , 0.80525273, ..., 0.36138955,
         0.31624836, 0.55308586],
        [0.55242676, 0.5738469 , 0.7176437 , ..., 0.37442863,
         0.33104813, 0.4594328 ],
        [0.58694977, 0.5446827 , 0.7393209 , ..., 0.36588195,
         0.3497949 , 0.44755676]],

       [[       nan,        nan,        nan, ..., 0.09767327,
         0.15716259, 0.19794455],
        [       nan, 0.35144722,        nan, ..., 0.13628796,
         0.11914784,        nan],
        [0.4051082 , 0.37889612, 0.35201168, ..., 0.12776695,
         0.1590535 , 0.22295249],
        ...,
        [       nan, 0.54239494, 0.42627606, ...,        nan,
                nan, 0.4412641 ],
        [0.5421717 ,        nan, 0.7211365 , ...,        nan,
         0.2999393 , 0.36796588],
        [0.46000457,        nan,        nan, ...,        nan,
                nan, 0.4003423 ]]], dtype=float32)
Coordinates:
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

## Modify the timeseries

The raw timeseries of temperature, precipitation as well as the run length is modified to adjust the range of all variables to avoid a division by zero.

In [15]:
da_rf_dekadal_modified = da_rf_dekadal + 1
da_ndvi_dekadal_modified =  da_ndvi_dekadal - (da_ndvi_dekadal.min() - 0.01)

## Calculate the Precipation Drought Index

### Group the modified timeseries using the `ip` parameter

The `ip` parameter determines to what extent past observations are considered. Longer IPs detect more severe drought events. The default `ip` used of **18 dekads** corresponds to the 6-month Standardized Precipitation Evapotranspiration Index.

For example, calculating the Precipitation Drought Index for the dekad `2011-01-11` using an interest period of `ip=3` requires rainfall data from the dekad `2010-12-21`, `2011-01-01` and `2011-01-11`.


**Each interest period is labelled using its end dekad.**

In [16]:
def get_dekad_no_in_month(date: np.datetime64) -> int:
    """
    Get the number of the dekad in a month that a date belongs to.
    Every month has three dekads, such that the first two dekads
    have 10 days (i.e., 1-10, 11-20), and the third is comprised of the
    remaining days of the month.

    Parameters
    ----------
    date : np.datetime64
        Date to check.

    Returns
    -------
    int
        Number of the dekad in a month.
    """
    timestamp = pd.Timestamp(date)
    year = timestamp.year
    month = timestamp.month

    first_day = datetime(year, month, 1)
    last_day = datetime(year, month, calendar.monthrange(year, month)[1])

    d1_start_date, d2_start_date, d3_start_date = pd.date_range(
        start=first_day, end=last_day, freq="10D", inclusive="left"
    )

    if d1_start_date <= timestamp < d2_start_date:
        return 1
    elif d2_start_date <= timestamp < d3_start_date:
        return 2
    else:
        return 3


def get_dekad_no_in_year(date: np.datetime64) -> int:
    """
    Get the number of the dekad in a year that a date belongs to.
    Every month has three dekads, such that the first two dekads
    have 10 days (i.e., 1-10, 11-20), and the third is comprised of the
    remaining days of the month (21-last day). Every year has 36 dekads.

    Parameters
    ----------
    date : np.datetime64
        Date to check.

    Returns
    -------
    int
        Number of the dekad in a year.
    """
    dekad_no_in_month = get_dekad_no_in_month(date=date)
    timestamp = pd.Timestamp(date)
    month = timestamp.month
    dekad_no_in_year = ((month - 1) * 3) + dekad_no_in_month
    return dekad_no_in_year


def get_interest_period(dekad: np.datetime64, ip: int) -> np.datetime64:
    """
    Get the start and end dekad of the interest period for a dekad.
    `dekad` will always be the end dekad of the interest period.

    Parameters
    ----------
    dekad : np.datetime64
        Dekad to get the interest period for.
        Will always be the end dekad of the interest period
    ip : int
        Number of dekads in an interest period.

    Returns
    -------
    np.datetime64
        Start and end dekad of the interest period.
    """
    year = pd.Timestamp(dekad).year
    dekad_no_in_year = get_dekad_no_in_year(dekad) - (ip - 1)
    while dekad_no_in_year <= 0:
        year -= 1
        dekad_no_in_year += 36

    month = (dekad_no_in_year - 1) // 3 + 1
    dekad_no_in_month = (dekad_no_in_year - 1) % 3 + 1
    if dekad_no_in_month == 1:
        day = 1
    elif dekad_no_in_month == 2:
        day = 11
    elif dekad_no_in_month == 3:
        day = 21

    start_dekad = np.datetime64(datetime(year, month, day), "ns")
    return (start_dekad, dekad)


def bin_by_interest_period(
    ds: xr.Dataset | xr.DataArray, ip: int
) -> dict[np.datetime64, xr.Dataset | xr.DataArray]:
    """
    Bin each dekad in the dataset by interest period.

    Parameters
    ----------
    ds : xr.Dataset | xr.DataArray
        Dataset to bin
    ip : int
        Number of dekads in an interest period.

    Returns
    -------
    list[tuple[np.datetime64, np.datetime64]]
        List of dekad ranges to bin
    """
    start_date = ds.dekad.min().values
    end_date = ds.dekad.max().values
    date_range = pd.date_range(
        start_date, end_date, freq=timedelta(days=1), inclusive="both"
    ).values
    dekads = np.unique(np.vectorize(get_dekad)(date_range))
    bins = list(get_interest_period(dekad=i, ip=ip) for i in dekads)
    binned_by_interest_period = {
        end: ds.sel(dekad=slice(start, end)) for start, end in bins
    }
    return binned_by_interest_period

In [17]:
# Group the modified rainfall and NDVI dekadal timeseries by the interest period.
# The key is the end dekad for each interest period.
# The values of the dictionary are the rainfall/NDVI data that belongs to the interest period
da_rf_binned_by_interest_period = bin_by_interest_period(ds=da_rf_dekadal_modified, ip=ip)
da_ndvi_binned_by_interest_period = bin_by_interest_period(ds=da_ndvi_dekadal_modified, ip=ip)

In [18]:
assert da_rf_binned_by_interest_period.keys() == da_ndvi_binned_by_interest_period.keys()

In [19]:
# Get the interest periods
interest_periods = list(da_rf_binned_by_interest_period.keys())

### Get the average values for each interest period

Get the actual average rainfall for each interest period.

In [20]:
da_rf_actual_avg_for_ip = xr.concat([da_rf_binned_by_interest_period[i].mean(dim="dekad").assign_coords(dekad=i).expand_dims({"dekad": 1}) for i in interest_periods], dim="dekad")
da_rf_actual_avg_for_ip

<xarray.DataArray 'rainfall' (dekad: 369, y: 49, x: 20)>
array([[[1.2920016, 1.       , 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.2998492, 1.2872081, 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.3341987, 1.3153791, 1.2823527, ..., 1.       , 1.       ,
         1.       ],
        ...,
        [1.       , 1.       , 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.       , 1.3435621, 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.       , 1.       , 1.       , ..., 1.       , 1.       ,
         1.       ]],

       [[1.1460009, 1.       , 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.1499245, 1.143604 , 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.1670994, 1.1576896, 1.1411763, ..., 1.       , 1.       ,
         1.       ],
...
        [4.4829135, 4.8552604, 5.346675 , ..., 4.028152 , 4.1134477,
         4.282174 ],
        [4.6004405, 5.016915 , 4.439429 , ..., 3.843149 , 4.099624 ,
         5.0744195],
        [5.1337695, 5.3067245, 5.2882113, ..., 4.2900314, 4.7853947,
         5.780925 ]],

       [[2.6244779, 2.3951905, 2.3968053, ..., 1.684229 , 1.6500363,
         1.6558833],
        [2.7968898, 2.6428645, 2.573401 , ..., 1.7260932, 1.6136231,
         1.8042831],
        [3.185643 , 3.0210404, 2.7688484, ..., 1.6584107, 1.8850683,
         1.9724536],
        ...,
        [4.883082 , 5.534899 , 6.0787687, ..., 3.7997673, 3.979492 ,
         4.070946 ],
        [5.0640593, 5.426829 , 5.0957103, ..., 3.6750257, 3.9406078,
         4.907514 ],
        [5.6229267, 5.80784  , 5.7753773, ..., 4.1103325, 4.55297  ,
         5.5532165]]], dtype=float32)
Coordinates:
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

Get the actual average NDVI for each interest period.

In [21]:
da_ndvi_actual_avg_for_ip = xr.concat([da_ndvi_binned_by_interest_period[i].mean(dim="dekad").assign_coords(dekad=i).expand_dims({"dekad": 1}) for i in interest_periods], dim="dekad")
da_ndvi_actual_avg_for_ip

<xarray.DataArray 'NDVI' (dekad: 369, y: 49, x: 20)>
array([[[17.69175435, 17.38121402, 17.53480489, ..., 17.23933069,
         17.27405185, 17.38286916],
        [17.66191126, 17.52106596, 17.57548274, ..., 17.23973035,
         17.24001278, 17.39169375],
        [17.71682341, 17.5235341 , 17.47557847, ..., 17.22129771,
         17.34771797, 17.38664416],
        ...,
        [17.97575081, 17.30578853, 17.4098508 , ...,         nan,
         17.57510043, 17.69738586],
        [17.80325306, 17.35917617,         nan, ..., 17.66969282,
         17.60861487, 17.75718785],
        [        nan, 17.73983384, 17.5248854 , ..., 17.7173161 ,
         17.65163247, 17.83139409]],

       [[17.70033352, 17.40200904, 17.48827502, ..., 17.23255305,
         17.27087312, 17.34030232],
        [17.61824691, 17.48972339, 17.57548274, ..., 17.23399391,
         17.23575035, 17.35941822],
        [17.68332166, 17.50333081, 17.47349405, ..., 17.23031976,
         17.30849003, 17.34632936],
...
        [17.72315429, 17.78032485, 17.86705343, ..., 17.63090616,
         17.56796265, 17.72440256],
        [17.73512559, 17.79810239, 17.84568233, ..., 17.66021712,
         17.57358224, 17.73026996],
        [17.81016754, 17.75562371, 17.92216968, ..., 17.6762904 ,
         17.61625588, 17.69706879]],

       [[17.63555064, 17.50432873, 17.47745216, ..., 17.25071343,
         17.30561403, 17.32942472],
        [17.53451728, 17.48860411, 17.53978226, ..., 17.29284333,
         17.28263055, 17.31809531],
        [17.53682446, 17.54072095, 17.47432973, ..., 17.29275347,
         17.31159539, 17.34392201],
        ...,
        [17.71029807, 17.76281607, 17.86138656, ..., 17.61885293,
         17.55359677, 17.71187877],
        [17.72159273, 17.79810239, 17.8489549 , ..., 17.64044364,
         17.55150123, 17.70975909],
        [17.79491139, 17.75562371, 17.92011659, ..., 17.65659398,
         17.59658996, 17.67951701]]])
Coordinates:
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

### Get the long term average for each interest period over the years of available data


In [22]:
# Group the interest periods by year by getting the dekad number in the year for the end dekad of
# the interest period. 
grouped_by_year = toolz.groupby(lambda dekad: get_dekad_no_in_year(date=dekad), interest_periods)

Get the long term average rainfall for each interest period.

In [23]:
long_term_avg_rf = {}
for dekad_no_in_year, interest_periods_list in grouped_by_year.items():
    long_term_avg_for_period = xr.concat([da_rf_binned_by_interest_period[i] for i in interest_periods_list], dim="dekad").mean(dim="dekad")
    long_term_avg_rf[dekad_no_in_year] = long_term_avg_for_period
    
da_rf_long_term_avg_for_ip = xr.concat([long_term_avg_rf[get_dekad_no_in_year(i)].assign_coords(dekad=i).expand_dims({"dekad": 1}) for  i in interest_periods], dim="dekad")
assert all(da_rf_actual_avg_for_ip.dekad.values == da_rf_long_term_avg_for_ip.dekad.values)
da_rf_long_term_avg_for_ip

<xarray.DataArray 'rainfall' (dekad: 369, y: 49, x: 20)>
array([[[2.7507236, 2.4500468, 2.3114686, ..., 1.3116618, 1.3749106,
         1.4813815],
        [2.9636745, 2.696412 , 2.5389245, ..., 1.3619807, 1.4258922,
         1.5707814],
        [3.2258933, 3.0636966, 2.77444  , ..., 1.4321268, 1.5358958,
         1.6278389],
        ...,
        [4.575324 , 4.641274 , 4.930497 , ..., 3.7197483, 3.7903256,
         3.9603279],
        [4.5558677, 4.706726 , 4.527445 , ..., 3.9371383, 3.7691438,
         4.130169 ],
        [4.896761 , 5.095007 , 5.0532374, ..., 3.8371177, 3.892133 ,
         4.443643 ]],

       [[2.6448786, 2.3647842, 2.2375066, ..., 1.2789761, 1.3397393,
         1.4437596],
        [2.8540728, 2.608467 , 2.4422157, ..., 1.3219744, 1.3905212,
         1.5332855],
        [3.0907307, 2.945651 , 2.6412635, ..., 1.3851899, 1.4933487,
         1.5863221],
...
        [3.1577332, 3.3235903, 3.5312552, ..., 3.0021243, 3.072863 ,
         3.2478325],
        [3.2210717, 3.2553668, 3.0311325, ..., 3.1251075, 3.2699466,
         3.539785 ],
        [3.3645403, 3.6043434, 3.547774 , ..., 3.2408574, 3.5295258,
         3.9360707]],

       [[2.125013 , 1.9563026, 1.9535156, ..., 1.5895246, 1.5515839,
         1.5692434],
        [2.2391195, 2.106691 , 2.105195 , ..., 1.6151519, 1.5337433,
         1.6512369],
        [2.4001937, 2.352779 , 2.228322 , ..., 1.547747 , 1.593801 ,
         1.7353896],
        ...,
        [3.1606774, 3.354914 , 3.5575788, ..., 3.050715 , 3.1302125,
         3.2656026],
        [3.2528756, 3.2548563, 3.0430722, ..., 3.1644168, 3.3211794,
         3.5670485],
        [3.4336734, 3.6690576, 3.6088066, ..., 3.2447107, 3.5791523,
         3.9889631]]], dtype=float32)
Coordinates:
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

Get the long term average NDVI for each interest period.

In [24]:
long_term_avg_ndvi = {}
for dekad_no_in_year, interest_periods_list in grouped_by_year.items():
    long_term_avg_for_period = xr.concat([da_ndvi_binned_by_interest_period[i] for i in interest_periods_list], dim="dekad").mean(dim="dekad")
    long_term_avg_ndvi[dekad_no_in_year] = long_term_avg_for_period
    
da_ndvi_long_term_avg_for_ip = xr.concat([long_term_avg_ndvi[get_dekad_no_in_year(i)].assign_coords(dekad=i).expand_dims({"dekad": 1}) for  i in interest_periods], dim="dekad")
assert all(da_ndvi_actual_avg_for_ip.dekad.values == da_ndvi_long_term_avg_for_ip.dekad.values)
da_ndvi_long_term_avg_for_ip

<xarray.DataArray 'NDVI' (dekad: 369, y: 49, x: 20)>
array([[[17.67613257, 17.47898275, 17.51501226, ..., 17.26328087,
         17.30324092, 17.35925169],
        [17.59188212, 17.47572793, 17.5224398 , ..., 17.28435213,
         17.28166078, 17.34515356],
        [17.5933563 , 17.52688673, 17.49626935, ..., 17.2931033 ,
         17.3298484 , 17.36096799],
        ...,
        [17.7855846 , 17.74821521, 17.77779357, ..., 17.71097668,
         17.54508629, 17.69600648],
        [17.78501437, 17.77164835, 17.81704747, ..., 17.70723513,
         17.51695644, 17.67206065],
        [17.79817776, 17.7819628 , 17.81874933, ..., 17.73244698,
         17.65391147, 17.72064414]],

       [[17.67719782, 17.47973945, 17.50675955, ..., 17.26133331,
         17.30488663, 17.35642035],
        [17.59478429, 17.47837664, 17.5160462 , ..., 17.28308119,
         17.27989929, 17.3422915 ],
        [17.59635374, 17.52448446, 17.49068504, ..., 17.29170576,
         17.32712935, 17.35877636],
...
        [17.72326079, 17.64001197, 17.73627201, ..., 17.62980342,
         17.51792254, 17.70909779],
        [17.69724945, 17.67357311, 17.77667608, ..., 17.61651422,
         17.49408474, 17.67753426],
        [17.70071507, 17.65602247, 17.79176496, ..., 17.63487642,
         17.58332085, 17.69820578]],

       [[17.63025963, 17.44543813, 17.45552182, ..., 17.25748   ,
         17.30019151, 17.34719212],
        [17.54469908, 17.44329038, 17.47882724, ..., 17.28080377,
         17.27502584, 17.33461954],
        [17.54216717, 17.47129534, 17.46157259, ..., 17.28605983,
         17.3214561 , 17.34157409],
        ...,
        [17.70884505, 17.6231967 , 17.72143245, ..., 17.61366907,
         17.50926631, 17.70052529],
        [17.68561228, 17.65593081, 17.77037692, ..., 17.60072827,
         17.48530285, 17.66947222],
        [17.68528461, 17.64016147, 17.77675823, ..., 17.62412393,
         17.57416991, 17.69241163]]])
Coordinates:
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

## Get the actual length of continous deficit or excess in each interest period

For rainfall and NDVI the run length is the number of successive dekads in an interest period below the long term average.
For temperature the run length is the number of successive dekads in an interest period above the long term average.

In [25]:
def get_no_data_mask(arr):
    """
    Check if all values in an array are NaN
    """
    return np.all(np.isnan(arr))

# From https://www.geeksforgeeks.org/maximum-consecutive-ones-or-zeros-in-a-binary-array/
def max_consecutive_ones(arr: np.ndarray) -> int:
    """
    Get the maximum number of successive ones in an array.

    Parameters
    ----------

    arr : np.ndarray
        Array to check

    Returns
    -------
    int
        Maximum number of consecutive ones in the input array.

    """

    n = len(arr)
    # initialize count
    count = 0
    # initialize max
    result = 0

    for i in range(0, n):
        # If 1 is found, increment count
        # and update result if count
        # becomes more.
        if arr[i] == 1:
            # increase count
            count += 1
            result = max(result, count)
        # Reset count if one is not found
        else:
            count = 0

    return result

In [26]:
%%time
# For each interest period get the maximum number of successive dekads below long term average rainfall in the interest period.
da_rf_run_length_ = []
for interest_period in interest_periods:
    # Get the dekads in the interest period
    ds = da_rf_binned_by_interest_period[interest_period]
    
    # Identify pixels which are empty for all dekads
    no_data_mask = xr.apply_ufunc(
        get_no_data_mask,
        ds,
        input_core_dims=[["dekad"]],
        vectorize=True,
        dask="allowed",
    )
    # Get the long term average for the dekads
    long_term_avg = da_rf_long_term_avg_for_ip.sel(dekad=ds.dekad)
    
    # Get the pixels where the rainfall is below the long term average rainfall
    mask = xr.where(ds < long_term_avg, 1, 0)
    
    # Get the maximum number of successive dekads below long term average rainfall.
    actual_run_length = xr.apply_ufunc(
        max_consecutive_ones,
        mask,
        input_core_dims=[["dekad"]],
        vectorize=True,
        dask="allowed",
    )
    # Modify the run length
    modified_run_length = (actual_run_length.max() + 1) - actual_run_length
    modified_run_length = modified_run_length.where(~no_data_mask)
    da_rf_run_length_.append(modified_run_length.assign_coords(dekad=interest_period).expand_dims({"dekad": 1}))

da_rf_actual_run_length = xr.concat(da_rf_run_length_, dim="dekad")
da_rf_actual_run_length

CPU times: user 6.66 s, sys: 0 ns, total: 6.66 s
Wall time: 6.59 s


<xarray.DataArray 'rainfall' (dekad: 369, y: 49, x: 20)>
array([[[ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        ...,
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.]],

       [[ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        ...,
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.]],

       [[ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        ...,
...
        ...,
        [10., 11., 11., ...,  7., 10., 10.],
        [10., 10., 10., ..., 11., 11., 10.],
        [10., 10., 10., ..., 11., 11., 12.]],

       [[ 7.,  7., 10., ...,  5.,  5.,  8.],
        [ 7.,  6.,  7., ...,  5.,  5.,  9.],
        [ 6.,  6.,  7., ...,  5.,  9.,  9.],
        ...,
        [ 9., 10., 10., ...,  6.,  9., 10.],
        [ 9.,  9.,  9., ..., 10., 10.,  9.],
        [ 9.,  9.,  9., ..., 10., 10., 11.]],

       [[ 6.,  6.,  9., ...,  5.,  5.,  8.],
        [ 6.,  5.,  6., ...,  5.,  5.,  9.],
        [ 5.,  5.,  6., ...,  5.,  9.,  9.],
        ...,
        [ 9., 10.,  9., ...,  5.,  8.,  9.],
        [ 9.,  9.,  9., ...,  9.,  9.,  8.],
        [ 9.,  9.,  8., ...,  9.,  9.,  9.]]])
Coordinates:
    spatial_ref  int32 6933
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

In [27]:
%%time
# For each interest period get the maximum number of successive dekads below long term average NDVI in the interest period.
da_ndvi_run_length_ = []
for interest_period in interest_periods:
    # Get the dekads in the interest period
    ds = da_ndvi_binned_by_interest_period[interest_period]
    
    # Identify pixels which are empty for all dekads
    no_data_mask = xr.apply_ufunc(
        get_no_data_mask,
        ds,
        input_core_dims=[["dekad"]],
        vectorize=True,
        dask="allowed",
    )
    # Get the long term average for the dekads
    long_term_avg = da_ndvi_long_term_avg_for_ip.sel(dekad=ds.dekad)
    
    # Get the pixels where the NDVI is below the long term average NDVI
    mask = xr.where(ds < long_term_avg, 1, 0)
    
    # Get the maximum number of successive dekads below long term average NDVI.
    actual_run_length = xr.apply_ufunc(
        max_consecutive_ones,
        mask,
        input_core_dims=[["dekad"]],
        vectorize=True,
        dask="allowed",
    )
    # Modify the run length
    modified_run_length = (actual_run_length.max() + 1) - actual_run_length
    modified_run_length = modified_run_length.where(~no_data_mask)
    da_ndvi_run_length_.append(modified_run_length.assign_coords(dekad=interest_period).expand_dims({"dekad": 1}))

da_ndvi_actual_run_length = xr.concat(da_ndvi_run_length_, dim="dekad")
da_ndvi_actual_run_length

CPU times: user 6.69 s, sys: 3.67 ms, total: 6.7 s
Wall time: 6.62 s


<xarray.DataArray 'NDVI' (dekad: 369, y: 49, x: 20)>
array([[[ 2.,  1.,  2., ...,  1.,  1.,  2.],
        [ 2.,  2.,  2., ...,  1.,  1.,  2.],
        [ 2.,  1.,  1., ...,  1.,  2.,  2.],
        ...,
        [ 2.,  1.,  1., ..., nan,  2.,  2.],
        [ 2.,  1., nan, ...,  1.,  2.,  2.],
        [nan,  1.,  1., ...,  1.,  1.,  2.]],

       [[ 3.,  1.,  2., ...,  1.,  1.,  2.],
        [ 2.,  2.,  3., ...,  1.,  1.,  2.],
        [ 3.,  1.,  1., ...,  1.,  2.,  2.],
        ...,
        [ 3.,  1.,  1., ...,  3.,  3.,  3.],
        [ 2.,  1.,  2., ...,  1.,  3.,  3.],
        [ 2.,  2.,  1., ...,  1.,  1.,  3.]],

       [[ 4.,  1.,  2., ...,  2.,  2.,  2.],
        [ 3.,  2.,  4., ...,  2.,  2.,  3.],
        [ 4.,  2.,  2., ...,  1.,  3.,  2.],
        ...,
...
        ...,
        [14., 15., 15., ..., 13., 15., 14.],
        [13., 14., 14., ..., 13., 15., 14.],
        [15., 14., 16., ..., 11., 13., 14.]],

       [[11., 12., 10., ..., 10., 14.,  9.],
        [ 8., 14., 14., ..., 14., 14., 10.],
        [10., 14., 13., ..., 13., 14.,  8.],
        ...,
        [14., 15., 15., ..., 13., 15., 14.],
        [13., 14., 14., ..., 13., 15., 13.],
        [15., 14., 16., ..., 10., 12., 13.]],

       [[11., 12., 10., ..., 11., 14.,  9.],
        [ 8., 14., 14., ..., 14., 15., 10.],
        [10., 14., 13., ..., 14., 14.,  8.],
        ...,
        [14., 15., 15., ..., 13., 15., 14.],
        [13., 14., 14., ..., 13., 15., 12.],
        [15., 14., 16., ..., 10., 12., 12.]]])
Coordinates:
    spatial_ref  int32 6933
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

### Get the long term average of continous deficit or excess in each interest period

In [28]:
# Get the long term average rainfall run length for each interest period 
long_term_avg_run_lenth_rf = {}
for dekad_no_in_year, interest_periods_list in grouped_by_year.items():
    long_term_avg_for_period = da_rf_actual_run_length.sel(dekad=interest_periods_list).mean(dim="dekad")
    long_term_avg_run_lenth_rf[dekad_no_in_year] = long_term_avg_for_period
    
da_rf_long_term_avg_run_length_for_ip = xr.concat([long_term_avg_run_lenth_rf[get_dekad_no_in_year(i)].assign_coords(dekad=i).expand_dims({"dekad": 1}) for  i in interest_periods], dim="dekad")
da_rf_long_term_avg_run_length_for_ip

<xarray.DataArray 'rainfall' (dekad: 369, y: 49, x: 20)>
array([[[7.72727273, 6.72727273, 6.72727273, ..., 3.        ,
         4.        , 6.        ],
        [7.81818182, 7.90909091, 7.36363636, ..., 3.54545455,
         5.09090909, 6.36363636],
        [7.45454545, 8.        , 7.36363636, ..., 3.81818182,
         5.90909091, 6.45454545],
        ...,
        [6.90909091, 7.36363636, 9.27272727, ..., 8.36363636,
         8.18181818, 8.90909091],
        [6.27272727, 6.63636364, 6.72727273, ..., 8.63636364,
         8.36363636, 8.81818182],
        [6.72727273, 7.09090909, 6.72727273, ..., 8.63636364,
         8.63636364, 8.63636364]],

       [[7.45454545, 6.63636364, 6.54545455, ..., 3.45454545,
         4.36363636, 6.36363636],
        [7.36363636, 7.72727273, 7.        , ..., 3.90909091,
         5.36363636, 6.72727273],
        [7.09090909, 7.63636364, 7.27272727, ..., 4.18181818,
         6.18181818, 6.72727273],
...
        [5.63636364, 6.90909091, 7.45454545, ..., 7.        ,
         7.09090909, 8.        ],
        [5.90909091, 5.90909091, 5.90909091, ..., 7.54545455,
         7.90909091, 7.72727273],
        [6.54545455, 6.45454545, 5.81818182, ..., 7.63636364,
         8.09090909, 7.81818182]],

       [[5.18181818, 6.27272727, 6.36363636, ..., 8.72727273,
         8.45454545, 8.63636364],
        [4.72727273, 5.81818182, 5.        , ..., 8.72727273,
         8.81818182, 9.27272727],
        [4.72727273, 5.09090909, 5.72727273, ..., 8.09090909,
         9.18181818, 9.72727273],
        ...,
        [5.36363636, 6.72727273, 6.90909091, ..., 6.36363636,
         6.63636364, 7.45454545],
        [5.90909091, 5.81818182, 5.90909091, ..., 6.90909091,
         7.45454545, 7.18181818],
        [6.54545455, 6.36363636, 5.81818182, ..., 7.        ,
         7.63636364, 7.27272727]]])
Coordinates:
    spatial_ref  int32 6933
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

In [29]:
# Get the long term average NDVI run length for each interest period 
long_term_avg_run_lenth_ndvi = {}
for dekad_no_in_year, interest_periods_list in grouped_by_year.items():
    long_term_avg_for_period = da_ndvi_actual_run_length.sel(dekad=interest_periods_list).mean(dim="dekad")
    long_term_avg_run_lenth_ndvi[dekad_no_in_year] = long_term_avg_for_period
    
da_ndvi_long_term_avg_run_length_for_ip = xr.concat([long_term_avg_run_lenth_ndvi[get_dekad_no_in_year(i)].assign_coords(dekad=i).expand_dims({"dekad": 1}) for  i in interest_periods], dim="dekad")
da_ndvi_long_term_avg_run_length_for_ip

<xarray.DataArray 'NDVI' (dekad: 369, y: 49, x: 20)>
array([[[ 7.90909091,  9.36363636,  9.72727273, ...,  7.09090909,
          9.18181818,  7.36363636],
        [ 7.63636364,  8.36363636,  8.45454545, ..., 11.09090909,
          9.63636364,  7.45454545],
        [ 7.72727273,  8.27272727,  8.72727273, ...,  9.54545455,
          7.27272727,  6.90909091],
        ...,
        [10.54545455,  9.90909091, 11.36363636, ..., 11.9       ,
         10.45454545, 11.18181818],
        [ 9.54545455, 10.54545455, 12.1       , ..., 10.72727273,
         11.09090909, 11.09090909],
        [10.8       ,  9.72727273, 11.45454545, ..., 10.90909091,
         10.81818182, 10.72727273]],

       [[ 7.63636364,  8.90909091,  9.18181818, ...,  7.09090909,
          9.18181818,  7.09090909],
        [ 7.36363636,  8.27272727,  8.36363636, ..., 10.72727273,
          9.36363636,  7.18181818],
        [ 7.45454545,  7.90909091,  8.36363636, ...,  9.36363636,
          7.18181818,  6.72727273],
...
        [ 9.18181818,  8.27272727, 10.45454545, ...,  8.54545455,
         10.45454545, 11.27272727],
        [ 8.36363636,  8.54545455, 10.09090909, ...,  8.        ,
         10.36363636, 10.09090909],
        [ 8.63636364,  8.18181818, 11.36363636, ...,  7.81818182,
          8.36363636, 10.18181818]],

       [[ 5.36363636,  8.09090909,  8.27272727, ...,  9.45454545,
         11.18181818,  6.54545455],
        [ 5.81818182,  7.81818182,  8.90909091, ..., 11.27272727,
         11.18181818,  7.36363636],
        [ 5.81818182,  7.72727273,  8.45454545, ..., 10.63636364,
          8.81818182,  7.09090909],
        ...,
        [ 9.09090909,  8.36363636, 10.36363636, ...,  8.36363636,
         10.54545455, 11.18181818],
        [ 8.27272727,  8.54545455, 10.18181818, ...,  7.90909091,
         10.63636364,  9.81818182],
        [ 8.54545455,  8.27272727, 11.27272727, ...,  7.90909091,
          8.27272727,  9.90909091]]])
Coordinates:
    spatial_ref  int32 6933
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

In [34]:
da_rf_actual_avg_for_ip

<xarray.DataArray 'rainfall' (dekad: 369, y: 49, x: 20)>
array([[[1.2920016, 1.       , 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.2998492, 1.2872081, 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.3341987, 1.3153791, 1.2823527, ..., 1.       , 1.       ,
         1.       ],
        ...,
        [1.       , 1.       , 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.       , 1.3435621, 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.       , 1.       , 1.       , ..., 1.       , 1.       ,
         1.       ]],

       [[1.1460009, 1.       , 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.1499245, 1.143604 , 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.1670994, 1.1576896, 1.1411763, ..., 1.       , 1.       ,
         1.       ],
...
        [4.4829135, 4.8552604, 5.346675 , ..., 4.028152 , 4.1134477,
         4.282174 ],
        [4.6004405, 5.016915 , 4.439429 , ..., 3.843149 , 4.099624 ,
         5.0744195],
        [5.1337695, 5.3067245, 5.2882113, ..., 4.2900314, 4.7853947,
         5.780925 ]],

       [[2.6244779, 2.3951905, 2.3968053, ..., 1.684229 , 1.6500363,
         1.6558833],
        [2.7968898, 2.6428645, 2.573401 , ..., 1.7260932, 1.6136231,
         1.8042831],
        [3.185643 , 3.0210404, 2.7688484, ..., 1.6584107, 1.8850683,
         1.9724536],
        ...,
        [4.883082 , 5.534899 , 6.0787687, ..., 3.7997673, 3.979492 ,
         4.070946 ],
        [5.0640593, 5.426829 , 5.0957103, ..., 3.6750257, 3.9406078,
         4.907514 ],
        [5.6229267, 5.80784  , 5.7753773, ..., 4.1103325, 4.55297  ,
         5.5532165]]], dtype=float32)
Coordinates:
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

## Calculate the scaled drought index

In [43]:
PDI = (da_rf_actual_avg_for_ip / da_rf_long_term_avg_for_ip) * np.sqrt((da_rf_actual_run_length / da_rf_long_term_avg_run_length_for_ip))
PDI_scaled = (PDI - PDI.min()) / (PDI.max() - PDI.min())
PDI_scaled

<xarray.DataArray 'rainfall' (dekad: 369, y: 49, x: 20)>
array([[[0.05322236, 0.04802559, 0.05225101, ..., 0.17468581,
         0.14042026, 0.10097422],
        [0.04779932, 0.053571  , 0.04255319, ..., 0.15218851,
         0.11675669, 0.09057467],
        [0.04539091, 0.04553169, 0.05383171, ..., 0.13759285,
         0.09750558, 0.08584209],
        ...,
        [0.01478737, 0.01310705, 0.00737672, ..., 0.01917986,
         0.01885611, 0.01543473],
        [0.01679771, 0.02717452, 0.0156864 , ..., 0.01625498,
         0.01863424, 0.01406346],
        [0.01280984, 0.01055728, 0.01171787, ..., 0.01726399,
         0.01670258, 0.01184278]],

       [[0.04862248, 0.05106518, 0.05578495, ..., 0.16595384,
         0.13758003, 0.10051891],
        [0.04404505, 0.04818337, 0.04686053, ..., 0.14890067,
         0.1166212 , 0.09016573],
        [0.0410576 , 0.04124387, 0.04930059, ..., 0.13565807,
         0.09817109, 0.08640045],
...
        [0.78100428, 0.7646847 , 0.76296384, ..., 0.53391184,
         0.65299057, 0.6377564 ],
        [0.76698143, 0.82937885, 0.78708867, ..., 0.61161454,
         0.60893465, 0.67045195],
        [0.77888937, 0.75620168, 0.80784997, ..., 0.65598985,
         0.65263249, 0.75779736]],

       [[0.57275915, 0.5138464 , 0.63103927, ..., 0.33674602,
         0.34382738, 0.43240378],
        [0.60781368, 0.49840732, 0.57728516, ..., 0.33983347,
         0.33236076, 0.45968195],
        [0.58889214, 0.54747329, 0.5471586 , ..., 0.35480115,
         0.50199903, 0.46720327],
        ...,
        [0.87386391, 0.87842294, 0.85097686, ..., 0.47202105,
         0.60270487, 0.59102455],
        [0.83804209, 0.90629544, 0.90311766, ..., 0.57120106,
         0.56144566, 0.62788079],
        [0.83757175, 0.82065812, 0.81802093, ..., 0.62087017,
         0.59605958, 0.67115364]]])
Coordinates:
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

In [44]:
VDI = (da_ndvi_actual_avg_for_ip / da_ndvi_long_term_avg_for_ip) * np.sqrt((da_ndvi_actual_run_length / da_ndvi_long_term_avg_run_length_for_ip))
VDI_scaled = (VDI - VDI.min()) / (VDI.max() - VDI.min())
VDI_scaled

<xarray.DataArray 'NDVI' (dekad: 369, y: 49, x: 20)>
array([[[0.14853945, 0.02985086, 0.11569112, ..., 0.0631559 ,
         0.03283968, 0.16088913],
        [0.15552321, 0.13986681, 0.13824793, ..., 0.01289959,
         0.02745097, 0.15922083],
        [0.1545341 , 0.04491908, 0.03859073, ..., 0.02809168,
         0.16293812, 0.17217424],
        ...,
        [0.10650603, 0.01972654, 0.00691687, ...,        nan,
         0.1051625 , 0.09506115],
        [0.11852296, 0.01376197,        nan, ..., 0.01634352,
         0.09766916, 0.09755174],
        [       nan, 0.02645792, 0.00697498, ..., 0.01490198,
         0.01589264, 0.10273642]],

       [[0.23125879, 0.03555502, 0.12385634, ..., 0.06308592,
         0.03277848, 0.16669606],
        [0.16087976, 0.1410182 , 0.21351671, ..., 0.01619721,
         0.03051172, 0.165127  ],
        [0.23785582, 0.04993709, 0.0434762 , ..., 0.0302953 ,
         0.16440237, 0.17619094],
...
        [0.63535801, 0.71685414, 0.61662723, ..., 0.63447707,
         0.61302628, 0.5558845 ],
        [0.64507579, 0.67141397, 0.60051507, ..., 0.66405236,
         0.61787628, 0.57121077],
        [0.69608026, 0.68904672, 0.60906193, ..., 0.56801745,
         0.61224233, 0.56552923]],

       [[0.76693468, 0.62680924, 0.54619926, ..., 0.53114804,
         0.55848573, 0.59316547],
        [0.59351187, 0.70646464, 0.65075584, ..., 0.55576025,
         0.58472843, 0.58839331],
        [0.68581104, 0.71293409, 0.63942999, ..., 0.57740345,
         0.6516584 , 0.52056503],
        ...,
        [0.63952941, 0.71190332, 0.62056105, ..., 0.64354419,
         0.60931523, 0.55872997],
        [0.64954372, 0.67227114, 0.59741551, ..., 0.66873477,
         0.60689851, 0.5510113 ],
        [0.70077544, 0.685006  , 0.61284574, ..., 0.56329037,
         0.6161398 , 0.5454172 ]]])
Coordinates:
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

In [74]:
VDI_scaled.to_dataframe().to_xarray()["NDVI"] == VDI_scaled

<xarray.DataArray 'NDVI' (dekad: 369, y: 49, x: 20)>
array([[[ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        ...,
        [ True,  True,  True, ..., False,  True,  True],
        [ True,  True, False, ...,  True,  True,  True],
        [False,  True,  True, ...,  True,  True,  True]],

       [[ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        ...,
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True]],

       [[ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        ...,
...
        ...,
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True]],

       [[ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        ...,
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True]],

       [[ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        ...,
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True]]])
Coordinates:
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933

Correlate the rainfall drought index with the vegetation drought index using Pearson's correlation coefficient. Find the time step at which the highest correlations are observed.

In [68]:
n = 10
pearsons_corr = xr.concat([xr.corr(PDI_scaled.shift(dekad=i), VDI_scaled, dim="dekad").assign_coords(lag=i).expand_dims({"lag": 1}) for i in range(0, n)], dim="lag")
time_lag = pearsons_corr.argmax(dim="lag")

In [69]:
time_lag

<xarray.DataArray (y: 49, x: 20)>
array([[7, 0, 9, 5, 0, 0, 0, 0, 0, 0, 0, 6, 5, 5, 6, 0, 5, 9, 9, 6],
       [0, 7, 9, 9, 0, 0, 0, 0, 0, 0, 9, 5, 0, 5, 7, 0, 9, 7, 6, 6],
       [0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 5, 6, 8, 6, 6, 8, 7, 9, 7],
       [0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 1, 4, 5, 5, 6, 0, 7, 9, 9],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 4, 9, 8, 9, 9, 7, 8, 7],
       [6, 0, 5, 0, 0, 1, 9, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 6, 6, 9],
       [5, 0, 0, 0, 0, 0, 9, 0, 0, 0, 4, 0, 0, 0, 6, 9, 0, 9, 6, 9],
       [5, 1, 0, 0, 0, 0, 0, 8, 1, 0, 0, 0, 0, 6, 0, 8, 7, 5, 8, 9],
       [1, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 8, 0, 7, 0, 9, 7, 0, 0, 9],
       [9, 0, 7, 0, 0, 1, 0, 9, 0, 0, 0, 0, 7, 6, 7, 8, 0, 0, 0, 9],
       [8, 6, 0, 0, 0, 1, 0, 9, 8, 1, 0, 3, 6, 6, 8, 6, 0, 6, 0, 0],
       [0, 6, 6, 0, 0, 0, 0, 1, 3, 1, 0, 8, 6, 0, 0, 0, 0, 9, 0, 9],
       [6, 1, 0, 0, 0, 0, 0, 6, 1, 0, 0, 7, 7, 0, 5, 3, 5, 5, 5, 9],
       [0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 7, 0, 6, 0, 0, 0, 0, 9],
       [0, 6, 0, 0, 0, 2, 0, 3, 0, 0, 0, 9, 6, 0, 7, 7, 9, 5, 6, 9],
       [0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 9, 8, 5, 0, 0, 0, 6, 0, 9, 9],
       [7, 0, 0, 0, 0, 0, 0, 0, 0, 9, 9, 9, 9, 7, 0, 0, 9, 0, 9, 9],
       [9, 6, 0, 0, 0, 0, 0, 0, 0, 0, 9, 9, 9, 0, 0, 8, 4, 0, 0, 0],
       [8, 5, 0, 0, 0, 9, 0, 0, 0, 7, 9, 8, 8, 0, 0, 0, 1, 9, 0, 0],
       [7, 5, 0, 0, 0, 0, 0, 0, 0, 0, 9, 9, 0, 0, 0, 0, 0, 0, 1, 5],
...
       [0, 0, 0, 5, 0, 1, 0, 0, 0, 0, 6, 0, 6, 0, 0, 0, 0, 1, 6, 0],
       [5, 0, 0, 0, 8, 0, 0, 0, 0, 0, 6, 5, 0, 0, 0, 7, 0, 6, 0, 0],
       [7, 6, 0, 0, 1, 0, 4, 0, 0, 0, 6, 0, 9, 0, 0, 0, 0, 0, 3, 4],
       [6, 6, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 6, 5, 5, 4],
       [6, 4, 0, 0, 0, 4, 1, 0, 0, 0, 6, 0, 6, 0, 0, 1, 9, 5, 5, 1],
       [0, 5, 0, 0, 1, 1, 0, 0, 6, 0, 0, 0, 0, 0, 0, 1, 6, 6, 5, 4],
       [0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 6, 0, 0, 6, 5, 8, 5],
       [0, 1, 1, 0, 0, 0, 8, 4, 9, 0, 9, 8, 0, 9, 0, 1, 8, 5, 7, 6],
       [1, 2, 0, 0, 1, 5, 1, 1, 0, 9, 9, 0, 9, 6, 6, 9, 7, 6, 9, 9],
       [1, 0, 5, 6, 0, 0, 1, 0, 4, 0, 0, 0, 9, 0, 0, 8, 6, 6, 9, 7],
       [6, 0, 6, 0, 0, 4, 1, 6, 0, 4, 9, 2, 0, 0, 9, 4, 7, 6, 8, 6],
       [5, 5, 5, 6, 0, 5, 5, 1, 1, 5, 0, 0, 3, 1, 0, 6, 9, 6, 9, 6],
       [0, 6, 9, 1, 0, 0, 1, 1, 5, 0, 0, 0, 0, 1, 1, 8, 6, 8, 1, 7],
       [5, 2, 0, 1, 1, 1, 3, 6, 0, 3, 1, 0, 0, 9, 1, 8, 5, 7, 5, 6],
       [6, 1, 1, 3, 1, 1, 1, 2, 0, 1, 4, 0, 0, 0, 1, 9, 6, 6, 9, 9],
       [6, 0, 1, 4, 6, 1, 6, 4, 0, 0, 0, 0, 0, 6, 1, 0, 1, 0, 0, 7],
       [4, 0, 0, 0, 7, 6, 4, 3, 0, 0, 0, 0, 0, 7, 0, 6, 0, 9, 7, 8],
       [4, 0, 0, 0, 6, 4, 2, 1, 0, 0, 0, 0, 0, 7, 7, 0, 6, 5, 9, 9],
       [6, 0, 1, 1, 4, 0, 0, 0, 0, 0, 0, 6, 6, 6, 0, 6, 1, 5, 7, 7],
       [1, 0, 5, 5, 5, 1, 1, 1, 0, 1, 1, 4, 0, 1, 0, 0, 0, 5, 0, 0]])
Coordinates:
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933